# Info

This notebook allows to evaluate the performance of ASR models.
The models under evaluation are defined directly inside `model-server/model-server-multi.py`

This notebook:
- Loads an evaluation dataset from Hugging Face (e.g. created by `model-trainer/1-create-datasets.ipynb`)
- Transcribes the audio via a remote server (`model-server-multi.py` - that provides a service to transcribe audios via the fine-tuned models) 
- Computes statistics (e.g. on WER) across all resulting transcriptions
- Prepares results files which can be further analyzed using a custom tool or through Nvidia Speech Explorer

# Requirements

- This notebook requires `model-server/model-server-multi.py` to serve the model(s) under evaluation.
Check that file to see how to use it

- The following libraries must be installed
```bashrc
sudo apt-get update
sudo apt-get install -y build-essential portaudio19-dev python3-dev libasound2-dev libjack-jackd2-dev pkg-config
```


# ToDo

- Assess if we can use: https://github.com/nvidia-nemo-speech/NeMo/tree/stable/tools/asr_evaluator 

# Parameters

## Datasets

In [ ]:
# Hugging Face evaluation datasets

# The notebook will use all splits within the datasets
# This (these) dataset(s) must be available on the Hugging Face Hub,
# (to be created by model-trainer/1-create-datasets.ipynb)
# if more than one dataset is specified in this variable, the model will be evaluated on the union of all datasets
HF_DATASET = [
    "luca-r/chiara_dataset-kto_20261112",
]

## Directories

In [ ]:
# Root directory for raw data (used as fallback to resolve audio paths)
DIR_DATA_RAW = "/home/luca/Development/chiara-speech2text_data/1-raw"

## Credentials

In [ ]:
# Credentials

# env file with configs and credentials
ENV_FILE = "/home/luca/Development/chiara-speech2text_envs/.env"

# Init notebook

In [ ]:
# Install packages
%pip install --quiet --upgrade pip setuptools wheel
%pip install --quiet huggingface_hub
%pip install --quiet datasets
%pip install --quiet torch
%pip install --quiet transformers
%pip install --quiet librosa
%pip install --quiet -U nemo_toolkit['asr']
%pip install --quiet --no-cache-dir pyaudio

In [ ]:
%pip install --upgrade --quiet lhotse

In [ ]:
from dotenv import load_dotenv

# Load the project env file before reading Firebase settings
load_dotenv(ENV_FILE, override=False)

In [ ]:
# Import utils

import os
import sys
from pathlib import Path

# Use the currently active notebook environment (the existing venv/kernel)
print(f"Python executable: {sys.executable}")
print(f"      Virtual env: {os.environ.get('VIRTUAL_ENV', 'not set')}")

repo_root = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "utils" / "utils.py").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the repository root containing utils/utils.py")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from utils.utils import (
    download_firebase_folders,
    my_normalizer,
)

try:
    del candidate, repo_root
except:
    pass

In [ ]:
# init
import os
import sys
import glob
import IPython.display as ipd

import torch
import librosa

from huggingface_hub import snapshot_download
from huggingface_hub import login, HfApi
from huggingface_hub.utils import HfHubHTTPError # Import HfHubHTTPError directly from utils

from transformers import WhisperForConditionalGeneration, WhisperProcessor

#from nemo.collections.asr.models import ASRModel

import requests

In [ ]:
import torch.utils.data
# Patch torch.utils.data.Sampler if it's missing __init__ or using object.__init__
# This fixes a TypeError in Lhotse/NeMo transcription with newer torch versions
if torch.utils.data.Sampler.__init__ is object.__init__:
    def sampler_init(self, data_source=None):
        pass
    torch.utils.data.Sampler.__init__ = sampler_init

In [ ]:
cuda_available = torch.cuda.is_available()
if cuda_available:
    print(f"CUDA device count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"CUDA device {i} name: {torch.cuda.get_device_name(i)}")
device = "cuda" if cuda_available else "cpu"

# cleanup
del cuda_available, i

In [ ]:
# Disable ipywidgets bars (they cause visualization errors with transforms library in notebooks)

import os
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]      = "1"

import datasets
datasets.disable_progress_bars()

In [ ]:
# Login to Hugging Face (to access models by "luca-r")

from huggingface_hub import login, HfApi
from huggingface_hub.utils import HfHubHTTPError

# HF login (access models by "luca-r")

# Load Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")
#print(f"I will use the HF token: {HF_TOKEN}")

# login to HF
login(token=HF_TOKEN) # Pass the token explicitly for consistency and clarity

# Verify login by attempting to retrieve user information
try:
    HfApi().whoami()
    print("Successfully logged in to Hugging Face Hub.")
except HfHubHTTPError as e:
    raise RuntimeError(f"Hugging Face login failed: {e}. Please ensure your 'hf_token' secret is valid and you have restarted the Colab runtime after setting it.")
except Exception as e:
    raise RuntimeError(f"An unexpected error occurred during Hugging Face login verification: {e}")

# cleanup
del HF_TOKEN

In [ ]:
# get today's date in format YYYYMMDD. this will be used to filename the ouptut from this notebook
from datetime import datetime
string_date = datetime.now().strftime("%Y%m%d")

## Download eval datasets from Hugging Face

In [ ]:
# Load all datasets from Hugging Face Hub

from datasets import load_dataset, DatasetDict

dataset_ids = [HF_DATASET] if isinstance(HF_DATASET, str) else HF_DATASET
if not dataset_ids:
    raise ValueError("HF_DATASET must contain at least one dataset ID")

datasets = {}
for dataset_id in dataset_ids:
    print(f"Loading evaluation dataset from Hugging Face: '{dataset_id}'...")
    hf_dataset = load_dataset(dataset_id)

    if isinstance(hf_dataset, DatasetDict):
        split_names = list(hf_dataset.keys())
        if not split_names:
            raise ValueError(f"Dataset '{dataset_id}' does not contain any splits")

        for split_name in split_names:
            dataset_key = f"{dataset_id}_{split_name}"
            print(f"dataset '{dataset_id}': found split '{split_name}'")
            datasets[dataset_key] = hf_dataset[split_name]
    else:
        datasets[dataset_id] = hf_dataset

    for dataset_key in [key for key in datasets if key == dataset_id or key.startswith(f"{dataset_id}_")]:
        dataset = datasets[dataset_key]
        print(f"{dataset_key} loaded successfully: {len(dataset):,} entries")
        print(f"Features: {dataset.features}")
        print(dataset)

In [ ]:
# Inspect first sample from evaluation dataset
if len(dataset) > 0:
    sample_preview = dataset[1]
    print("Sample 1 columns:", list(sample_preview.keys()))
    print(f"Sample 1 audio: {sample_preview.get('audio')}")
    print(f"Sample 1 text:  {sample_preview.get('text')}")
    if 'input_length' in sample_preview and sample_preview.get('input_length') is not None:
        print(f"Sample 1 duration: {sample_preview.get('input_length'):.2f}s")

# Transcribe data

## Define useful functions

In [ ]:
from flask import logging
import requests
import os
import tempfile

# the following script serves the endpoints below
# model-server/model-server-multi.py

def transcribe_multi(wav_file: str, base_url: str = "http://127.0.0.1:9100") -> dict:
    with open(wav_file, "rb") as f:
        files = {"wav": f}
        response = requests.post(f"{base_url}/transcribe_multi", files=files)
    response.raise_for_status()
    return response.json()


def _resolve_audio_file(audio_item, base_dir=None):
    """Resolve an audio item from the Hugging Face dataset to a valid local file path."""
    if isinstance(audio_item, dict):
        if audio_item.get("path") and os.path.exists(audio_item["path"]):
            return audio_item["path"]
        if audio_item.get("bytes"):
            tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
            tmp.write(audio_item["bytes"])
            tmp.close()
            return tmp.name
        if audio_item.get("path"):
            audio_item = audio_item["path"]

    if isinstance(audio_item, str):
        if os.path.exists(audio_item):
            return audio_item
        if base_dir:
            cand = os.path.join(base_dir, audio_item)
            if os.path.exists(cand):
                return cand
            cand = os.path.join(base_dir, os.path.basename(audio_item))
            if os.path.exists(cand):
                return cand
    return str(audio_item) if audio_item is not None else None

## Transcribe 1 sample

In [ ]:
# select audio file (use sample 0 from HF eval dataset if available, or fall back to asset sample)
if "dataset" in globals() and len(dataset) > 0:
    sample_entry = dataset[0]
    sample_audio = sample_entry.get("audio") or sample_entry.get("audio_data") or sample_entry.get("audio_path")
    wav_file = _resolve_audio_file(sample_audio, base_dir=DIR_DATA_RAW) if "DIR_DATA_RAW" in globals() else str(sample_audio)
    print(f"Selected sample 0 from eval dataset: {wav_file}")
    print(f"Ground truth text: {sample_entry.get('text') or sample_entry.get('ground_truth')}")
else:
    #wav_file = '/home/luca/Development/chiara-speech2text_private-v3/assets/audios/1-chiara/chiara-1.wav'
    #wav_file = '/home/luca/Development/chiara-speech2text_private-v3/assets/audios/1-chiara/chiara-2.wav'
    #wav_file = '/home/luca/Development/chiara-speech2text_private-v3/assets/audios/1-chiara/chiara-3.wav'
    wav_file = '/home/luca/Development/chiara-speech2text_private-v3/assets/audios/1-chiara/chiara-4.wav'
    #wav_file = '/home/luca/Development/chiara-speech2text_private-v3/assets/audios/1-chiara/chiara-5.wav'

# show ipy widget to play audio
#ipd.Audio(wav_file)

In [ ]:
response = transcribe_multi(wav_file)
#print(f"    Vanilla: {response.get('transcript_whisper_vanilla')}")
print(f"Finetuned 1: {response.get('transcript_whisper_finetuned1')}")
print(f"Finetuned 2: {response.get('transcript_whisper_finetuned2')}")

In [ ]:
# cleanup
del wav_file

## Transcribe dataset(s)

In [ ]:
# Set folder for output results
DIR_RESULTS_OUTPUT = f"results_{string_date}"

!rm -rf {DIR_RESULTS_OUTPUT}
!mkdir  {DIR_RESULTS_OUTPUT}

In [ ]:
# this can take ~1h30 for ~6500 samples per dataset

# @todo: check if writing on the json file only at the end of the loop is better than writing line by line (it should be faster, but it will use more memory)

import json, os, csv
import librosa

idx_stop = None # process all samples
#idx_stop = 10 # only process the first N samples: for testing purposes

for dataset_id, dataset in datasets.items():   

    # split_name: only keep the string after the last underscore, if present
    split_name = dataset_id.replace("/", "-")
    #if "_" in split_name: split_name = split_name.split("_")[-1]

    #
    results_filename        = f'{DIR_RESULTS_OUTPUT}/{split_name}_results.json'
    normalizations_filename = f'{DIR_RESULTS_OUTPUT}/{split_name}_normalizations.json'
    errors_filename         = f'{DIR_RESULTS_OUTPUT}/{split_name}_errors.csv'
    global_count = 0
    total_samples = len(dataset)
    normalizations = []

    print(f"\nTranscribing data: '{dataset_id}' ({total_samples:,} samples)")

    with open(results_filename, 'w', encoding='utf-8') as file_results, open(errors_filename, 'w', newline='', encoding='utf-8') as file_errors:
        errors_writer = csv.writer(file_errors)
        errors_writer.writerow(["sample_idx", "wav_filename", "error"])

        for idx, sample in enumerate(dataset, 1):
            if idx_stop is not None and idx > idx_stop:
                break  # only process the first N samples: for testing purposes

            audio_val = sample.get("audio") or sample.get("audio_data") or sample.get("audio_path")
            wav_file = _resolve_audio_file(audio_val, base_dir=DIR_DATA_RAW) if "DIR_DATA_RAW" in globals() else str(audio_val)

            if not wav_file or not os.path.exists(wav_file):
                error = f"Audio file not found: {audio_val}"
                print(f"[{idx}/{total_samples}] WARNING: Skipping sample: {error}")
                errors_writer.writerow([idx, str(audio_val), error])
                continue

            # Extract ground truth text
            ground_truth = sample.get("ground_truth") or sample.get("text")

            if not ground_truth or not isinstance(ground_truth, str) or not ground_truth.strip():
                error = "Missing ground truth transcription"
                print(f"[{idx}/{total_samples}] WARNING: Skipping {wav_file}: {error}")
                errors_writer.writerow([idx, wav_file, error])
                continue

            ground_truth = ground_truth.strip()

            # normalize transcript
            transcript_normalized = my_normalizer(
                ground_truth,
                audio_filename = wav_file,
                verbose        = False,
            )

            # Record normalization if it changed the ground truth
            if transcript_normalized is not None and transcript_normalized != ground_truth:
                normalizations.append(
                    {
                        " audio_filename": wav_file,
                        "  original_text": ground_truth,
                        "normalized_text": transcript_normalized,
                    }
                )
                ground_truth = transcript_normalized

            # Get duration: use input_length from dataset if present, else librosa
            duration = sample.get("input_length")
            if duration is None:
                try:
                    duration = librosa.get_duration(path=wav_file)
                except Exception:
                    duration = None

            global_count += 1
            print(f"   Processing: {idx}/{total_samples} - {wav_file}")

            try:
                # Transcribe audio using the model server
                response = transcribe_multi(wav_file)

                # Extract transcripts from the response
                #transcript_whisper_vanilla        = response.get("transcript_whisper_vanilla")
                transcript_whisper_finetuned1     = response.get("transcript_whisper_finetuned1")
                transcript_whisper_finetuned2     = response.get("transcript_whisper_finetuned2")
                #transcript_whisper_finetuned2_llm = response.get("transcript_whisper_finetuned2_llm")

                # Record results in JSONL format
                line = {
                    "audio_filepath":       wav_file,
                    "duration":             duration,
                    "text":                 ground_truth,
                    #"pred_text_vanilla":    transcript_whisper_vanilla,
                    "pred_text_finetuned1": transcript_whisper_finetuned1,
                    "pred_text_finetuned2": transcript_whisper_finetuned2,
                    #"pred_text_finetuned2_llm": transcript_whisper_finetuned2_llm
                }
                file_results.write(json.dumps(line, ensure_ascii=False) + '\n')

            except Exception as e:
                error = str(e)
                print(f"Error processing {wav_file}: {error}")
                errors_writer.writerow([idx, wav_file, error])

    with open(normalizations_filename, "w", encoding="utf-8") as file_normalizations:
        json.dump(normalizations, file_normalizations, ensure_ascii=False, indent=2)

    print(f"   Results saved to {results_filename} ({global_count} total files processed)")
    print(f"   Errors saved to {errors_filename}")
    if normalizations:
        print(f"   Normalizations saved to {normalizations_filename} ({len(normalizations)} substitutions)")

In [ ]:
# cleanup
try:
    del ground_truth, line
except NameError:
    pass

In [ ]:
# Convert all "text" to lower-case (for compatibility with Nvidia Nvidia Speech Explorer)
# (this step shall be already done in the 1-create-datasets.ipynb notebook. check whether it is necessary to do it again here, or if it can be skipped)

import json

# Read all lines, lowercase the "text" field, and write back
with open(results_filename, "r") as f:
    lines = f.readlines()

with open(results_filename, "w") as f:
    for line in lines:
        line = line.strip()
        if not line:
            continue
        entry = json.loads(line)
        entry["text"] = entry["text"].lower()
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Lowercased 'text' in {len(lines)} entries.")

# Analyze results...

## ...with this notebook

In [ ]:
# specify here again to allow running notebook from this cell (without the previous inference)
# if one already computed the results in previous iterations

# set manually
#string_date = "20261018"

# get today's date in format YYYYMMDD
from datetime import datetime
string_date = datetime.now().strftime("%Y%m%d")
DIR_RESULTS_OUTPUT = f"results_{string_date}"

# set file to be analyzed
results_filename = [
    "/home/luca/Development/chiara-speech2text_private-v3/models-evaluator/results_20261113/luca-r-chiara_dataset-kto_20261112_train_results.json",
    "/home/luca/Development/chiara-speech2text_private-v3/models-evaluator/results_20261113/luca-r-chiara_dataset-kto_20261112_test_results.json",
    "/home/luca/Development/chiara-speech2text_private-v3/models-evaluator/results_20261113/luca-r-chiara_dataset-kto_20261112_eval_results.json",
]

In [ ]:
print(f"Analyzing results from {len(results_filename)} files:")
for result_filename in results_filename:
    print(f"   {result_filename}")

In [ ]:
# Compute WER for each results file and across all samples

import json
import os
from jiwer import wer

model_columns = {
    #"vanilla":      "pred_text_vanilla",
    "finetuned1":   "pred_text_finetuned1",
    "finetuned2":   "pred_text_finetuned2",
}
all_wers = {model_name: [] for model_name in model_columns}
file_summaries = []

def format_wer(value):
    return f"{value:.2f}" if value is not None else "N/A"

for result_filename in results_filename:
    file_wers = {model_name: [] for model_name in model_columns}

    with open(result_filename, "r", encoding="utf-8") as results_file:
        for line in results_file:
            if not line.strip():
                continue

            record = json.loads(line)
            ground_truth = record["text"]
            for model_name, prediction_column in model_columns.items():
                file_wers[model_name].append(wer(ground_truth, record[prediction_column]))

    sample_count = len(next(iter(file_wers.values()), []))
    file_average_wer = {
        model_name: round(sum(scores) / len(scores), 2) if scores else None
        for model_name, scores in file_wers.items()
    }
    print(f"\nWER for {result_filename} ({sample_count} samples):")
    for model_name, average_wer in file_average_wer.items():
        print(f"- {model_name}: {format_wer(average_wer)}")

    file_summaries.append({
        "file": result_filename,
        "sample_count": sample_count,
        "average_wer": file_average_wer,
    })
    for model_name, scores in file_wers.items():
        all_wers[model_name].extend(scores)

total_samples = len(next(iter(all_wers.values()), []))
#avg_wer_whisper_vanilla = sum(all_wers["vanilla"]) / total_samples if total_samples else None
avg_wer_whisper_finetuned1 = sum(all_wers["finetuned1"]) / total_samples if total_samples else None
avg_wer_whisper_finetuned2 = sum(all_wers["finetuned2"]) / total_samples if total_samples else None

average_wer = {
    #"vanilla":      round(avg_wer_whisper_vanilla, 2)       if avg_wer_whisper_vanilla      is not None else None,
    "finetuned1":   round(avg_wer_whisper_finetuned1, 2)    if avg_wer_whisper_finetuned1   is not None else None,
    "finetuned2":   round(avg_wer_whisper_finetuned2, 2)    if avg_wer_whisper_finetuned2   is not None else None,
}
print(f"\nAverage WER across all splits ({total_samples} samples):")
for model_name, average in average_wer.items():
    print(f"- {model_name}: {format_wer(average)}")

summary = {
    "files": file_summaries,
    "total_samples": total_samples,
    "average_wer": average_wer,
}
os.makedirs(DIR_RESULTS_OUTPUT, exist_ok=True)
summary_filename = f"{DIR_RESULTS_OUTPUT}/{string_date}_results_wer.json"
with open(summary_filename, "w", encoding="utf-8") as summary_file:
    json.dump(summary, summary_file, indent=4)
print(f"\nWER summary saved to {summary_filename}")

In [ ]:
# Plot average WER by dataset for each fine-tuned model

import os
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('dark_background')

model_names = ['chiara_whisper_sft\n20261015', 'chiara_whisper_sft+kto\n20261026']
model_keys = ['finetuned1', 'finetuned2']
colors = ['blue', 'orange']

dataset_labels = []
wers_by_model = {key: [] for key in model_keys}

for summary in file_summaries:
    label = os.path.basename(summary['file']).replace('_results.json', '')
    dataset_labels.append(label)
    for key in model_keys:
        wers_by_model[key].append(summary['average_wer'].get(key))

x = np.arange(len(dataset_labels))
width = 0.35

plt.figure(figsize=(12, 5))
for i, (key, name, color) in enumerate(zip(model_keys, model_names, colors)):
    offset = (i - (len(model_keys) - 1) / 2) * width
    bars = plt.bar(x + offset, wers_by_model[key], width, label=name.replace('\n', ' '), color=color)
    for bar, value in zip(bars, wers_by_model[key]):
        if value is not None:
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{value:.2f}',
                ha='center',
                va='bottom',
            )

plt.xlabel('Dataset')
plt.ylabel('Word Error Rate (WER)')
plt.title('Average WER by dataset and model')
plt.xticks(x, dataset_labels, rotation=20, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Merge all results files listed in results_filename into one JSON file

import json
import os
from pathlib import Path

merged_records = []
for result_path in results_filename:
    with open(result_path, "r", encoding="utf-8") as result_file:
        content = result_file.read().strip()

    if not content:
        continue

    try:
        parsed_results = json.loads(content)
    except json.JSONDecodeError:
        parsed_results = [
            json.loads(line)
            for line in content.splitlines()
            if line.strip()
        ]

    if isinstance(parsed_results, list):
        merged_records.extend(parsed_results)
    elif isinstance(parsed_results, dict):
        merged_records.append(parsed_results)
    else:
        raise ValueError(f"Unsupported JSON data in results file: {result_path}")

# 
os.makedirs(DIR_RESULTS_OUTPUT, exist_ok=True)
merged_results_prediction_filename = f"{DIR_RESULTS_OUTPUT}/{string_date}_results_predictions.json"
with open(merged_results_prediction_filename, "w", encoding="utf-8") as merged_file:
    json.dump(merged_records, merged_file, ensure_ascii=False, indent=2)
print(f"Merged {len(merged_records)} records from {len(results_filename)} files into {merged_results_prediction_filename}")

# JSONL in the same format as the per-split *_results.json files (for Nvidia Speech Explorer)
first_results_path = Path(results_filename[0])
merged_stem = f"{string_date}_results_splits"
#merged_jsonl_filename = str(first_results_path.with_name(f"{merged_stem}.json"))
merged_jsonl_filename = f"{DIR_RESULTS_OUTPUT}/{string_date}_results_splits.json"
with open(merged_jsonl_filename, "w", encoding="utf-8") as merged_jsonl_file:
    for record in merged_records:
        merged_jsonl_file.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Merged JSONL saved to {merged_jsonl_filename}")

In [ ]:
merged_results_filename = merged_results_prediction_filename

In [ ]:
# convert the merged JSON file to a more human-readable JSON file
# this can be reviewed more easily by a human

import re
import json
import os

output_filename = f'{DIR_RESULTS_OUTPUT}/{string_date}_results_human-readable.json'

# remove output_filename if it already exists, to avoid confusion with previous runs
if os.path.exists(output_filename):
    os.remove(output_filename)

with open(merged_results_filename, 'r', encoding='utf-8') as infile:
    data = json.load(infile)

if not isinstance(data, list):
    raise ValueError(f"Expected a JSON array in {merged_results_filename}")

# remove fields to make it more readable
fields_to_remove = ['audio_filepath', 'duration']

for item in data:
    for field in fields_to_remove:
        item.pop(field, None)

# save to new json file
with open(output_filename, 'w', encoding='utf-8') as outfile:
    json.dump(data, outfile, indent=4, ensure_ascii=False)

# rename fields to make them shorter
with open(output_filename, 'r', encoding='utf-8') as f:
    content = f.read()

# note: the extra spaces in the replacement strings are intentional, to make the columns align better when viewed in a text editor
content = re.sub(r'\btext\b',                   'ground-truth', content)
content = re.sub(r'\bpred_text_vanilla\b',      '     vanilla', content)
content = re.sub(r'\bpred_text_finetuned1\b',   '  finetuned1', content)
content = re.sub(r'\bpred_text_finetuned2\b',   '  finetuned2', content)

# save the modified content back to the file
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(content)

#
print(f"Readable results saved to {output_filename}")

## ...with Nvidia Nvidia Speech Explorer

The .jsonl file generated above can be used with Nvidia Speech Explorer tool
- https://docs.nvidia.com/nemo-framework/user-guide/latest/nemotoolkit/tools/speech_data_explorer.html 
- https://docs.nvidia.com/nemo-framework/user-guide/latest/nemotoolkit/tools/comparison_tool.html 

```bash
# Install nvidia Nemo tool from here: https://github.com/nvidia-nemo-speech/Speech/tree/main 

# Show stats for model 2
python3 /home/luca/Development/nvidia-nemo-speech/tools/speech_data_explorer/data_explorer.py                           \
    /home/luca/Development/chiara-speech2text_private-v3/models-evaluator/results_20261113/20261113_results_splits.json \
    --names_compared pred_text_finetuned1 pred_text_finetuned2                                                          \
    --show_statistics pred_text_finetuned1
```

## ...with custom analysis-tool

- Run Nvidia Nvidia Speech Explorer (as shown above)
- In the "Statistics" page, click on "Download Vocabulary" (at the very bottom of the page)
- Launch the tool pointing at the downloaded .csv file, e.g.:

```bash
python /home/luca/Development/chiara-speech2text_private-v3/models-evaluator/analysis-tool/launch.py \
    /home/luca/Development/chiara-speech2text_private-v3/models-evaluator/results_20261110/20261111_sde_vocab_dec.csv
```

# Best model

The best performing model can be pushed to Hugging Face with the script: `utils/push-model-to-hf-hub.py`

# Cleanup

In [ ]:
# Empty GPU RAM
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU RAM emptied.")
else:
    print("No GPU available, no GPU RAM to empty.")